# LASSI - Power Efficiency

***OLLAMA API*** Jupyter Notebook version

+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

Power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

## Setup

In [1]:
import os

# Import LASSI modules
import lassi.llm_config
import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary

# Enable the autoreload extension
%load_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline
%aimport lassi.get_power

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [2]:
def experimental_setup():
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.

    -- matrix-rotate ["10000", "1"]
    -- jacobi [""] (none required)
    -- DNU -- mallocFree ["10000"]
    -- layout ["1"]
    -- atomicCost ["1"]
    -- dense-embedding ["10000", "8", "1"]
    -- pathfinder ["10000", "1000", "1000"]
    -- bsearch ["10000", "1"]
    -- entropy ["10000", "1024", "1"]
    -- colorwheel ["10000", "8", "1"]
    -- randomAccess ["1"]
    -- lid-driven-cavity [""]
    """
    
    app_folder_name = "lid-driven-cavity"

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    test_code_first = False  # True or False
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context = True  # True or False

    # TOGGLE USE OF POWER MEASUREMENT DURING EXE
    power_measure = False  # True or False

    # TOGGLE ENERGY EFFICIENCY SELF-CORRECTION FEEDBACK
    yes_power_feedback = False  # True or False

    # ASSIGN CODE COMPILE COMMAND
    if source_lang == "CUDA":
        code_compiler = "nvcc"
        code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_80 -O3"
        # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # Linux_x86_64 or Linux_aarch64
        code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        context_knowledge_selection = "omp"

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1"]
        exe_input_parameters = [""]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL
    
    # -- FROM LMSTUDIO
    # model = "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF/"
    # model = "TheBloke/CodeLlama-13B-Instruct-GGUF"
    # model = "lmstudio-community/starcoder2-15b-instruct-v0.1-GGUF" - couldn;t generate anything
    # model = "lmstudio-community/stable-code-instruct-3b-GGUF" - couldn't compile, but 'worked'
    # model = "microsoft/Phi-3-mini-4k-instruct-gguf" - generated garbage, and slow
    # model = "lmstudio-community/codegemma-1.1-7b-it-GGUF" - generated code, faster, but not compiled -- more tries
    # model = "lmstudio-community/Codestral-22B-v0.1-GGUF"  # - nice large context size 32768 -- load it on aiop01 at 28 gpu layers

    # -- FROM OLLAMA
    model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    model_coder = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "qwq:latest"  # (max token = 32768)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 128000
    num_ctx_coder = 128000

    # ASSIGN LLM API SOURCE:
    if "GGUF" in model:
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "02test_betterpower_withcontext"  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    return (app_folder_name, source_lang, target_lang, model, model_coder, num_ctx, num_ctx_coder, llm_source, use_context,
            context_knowledge_selection, code_compiler, code_compiler_kwds, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, test_code_first, yes_power_feedback)

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [3]:
prompts = PromptDictionary()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, source_lang, target_lang, model, model_coder, num_ctx, num_ctx_coder,
    llm_source, use_context, context_knowledge_selection, code_compiler,
    code_compiler_kwds, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, test_code_first, yes_power_feedback
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))


# LOAD INPUT CODE TO TRANSLATE
input_code_folder = "HeCBench/" + app_folder_name + "/"
input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    input_code_extension = "cu"
source_file_directory = "source_codes/" + input_code_folder
source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()

# SPECIFY OUTPUT CODE SAVE
directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment


# CREATE METADATA RECORD CONTENT
meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
meta_data_file += "\nApplication: " + app_folder_name
meta_data_file += "\n\nEE Refactoring: " + system_prompt_selection_index
# meta_data_file += "\nTranslate from: " + input_code_extension
# meta_data_file += "\nTranslate to: " + translated_code_extension
meta_data_file += "\n\nLLM Source: " + llm_source
meta_data_file += "\nModel: " + model
meta_data_file += "\nModel coder: " + model_coder
meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
meta_data_file += "\nCode file template: " + directory + "/" + file_name
meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
meta_data_file += "\n\nPower measurement: " + str(power_measure)

# Display metadata
print("\n" + meta_data_file + "\n")

print("\n\n" + color.BOLD + "***** BEGIN THE LASSI PIPELINE*****" + color.END + "\n\n")

# RUN PIPELINE
complete = lassi.pipeline.auto_code_llmgeneration_pipeline(
    prompts,
    this_llm_system_prompt,
    this_llm_refactoredcode_prompt,
    llm_source, 
    model,
    model_coder,
    num_ctx,
    num_ctx_coder,
    # arrayormatrix_size_range,
    # thread_count_range,
    use_context,
    input_context,
    input_code,
    # target_code,
    target_lang,
    input_code_extension,
    source_lang,
    code_compiler,
    code_compiler_kwds,
    directory,
    file_name,
    source_file_path,
    display_response,
    test_with_error,
    generate_test_case,
    yes_execute,
    exe_input_parameters,
    meta_data_file,
    power_measure, 
    test_code_first,
    yes_power_feedback
)

# Reset to original working directory
os.chdir(original_home_directory)

if complete:
    print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
else:
    print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)


Unloading Ollama server...

Unload LLM:
 {"model":"deepseek-r1:70b","created_at":"2025-02-07T07:19:02.651017963Z","response":"","done":true,"done_reason":"unload"}
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   167  100   122  100    45  48878  18028 --:--:-- --:--:-- --:--:-- 83500


Unloading Ollama server...

Unload LLM:
 {"model":"deepseek-coder-v2:16b-lite-instruct-fp16","created_at":"2025-02-07T07:19:02.65975811Z","response":"","done":true,"done_reason":"unload"}
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   216  100   146  100    70   113k  55776 --:--:-- --:--:-- --:--:--  210k


***** EXP



----- CONTEXT KNOWLEDGE SUMMARY -----
 (generated by the LLM)



**Answer:**

The CUDA programming model offers a powerful paradigm for parallel computing on NVIDIA GPUs, with several key features and optimizations to consider:

1. **Thread Hierarchy and Execution Model:**
   - Kernels are executed as grids of thread blocks, each containing multiple threads. Each thread has unique IDs (threadIdx), enabling parallel processing.
   - Thread blocks organize computation efficiently, allowing for data partitioning and independent execution.

2. **Memory Management:**
   - **Shared Memory:** On-chip memory shared among threads within a block, ideal for frequent access.
   - **Registers:** Per-thread on-chip memory for fast access to local variables.
   - **Local Memory:** Off-chip but useful when registers or shared memory are insufficient.

3. **Synchronization:**
   - **Barriers:** Ensure all threads in a block reach a point before proceeding, preventing data races.
   - **Cooperative Groups:** Offer flexible groupings beyond thread blocks for complex parallelism tasks.

4. **Memory Operations:**
   - **Memory Fence Functions:** Synchronize memory operations to ensure consistency across threads or blocks.

5. **Compute Capability and Architecture:**
   - Each GPU architecture (e.g., Ampere with compute 8.0) has specific features; check compatibility for optimal use.
   - Be aware of deprecated architectures (e.g., Tesla, Fermi) when targeting older hardware.

6. **Asynchronous Programming and Streams:**
   - Use streams to overlap computation and data transfer, enhancing performance but requiring careful dependency management.
   - Hyper-Q allows multiple streams on the same SMX for improved utilization.

7. **Occupancy and Optimization:**
   - Balance occupancy (active warps per SM) with resource usage; higher occupancy hides latency.
   - Tools like nvvp can help analyze and optimize occupancy.

8. **Cache Policies:**
   - Optimize data access patterns by setting cache preferences, either for larger blocks or shared memory efficiency.
   - Profiling tools can help determine optimal settings.

9. **Unified Memory:**
   - Simplifies memory management with a single allocation for host and device use.
   - Consider performance trade-offs; manual transfer may still be beneficial for optimization.

10. **Further Learning and Application:**
    - Explore cooperative groups through examples to understand use cases beyond traditional thread blocks.
    - Study asynchronous operations with different scopes (thread, block, device, system) using examples.
    - Use profiling tools to analyze memory access patterns and cache policy impact.

By mastering these concepts and experimenting with practical examples, you can effectively leverage CUDA's capabilities for efficient GPU programming.

GPU Memory Status:
 GPU 0: 23859 MiB used, 17100 MiB free, 40960 MiB total
 GPU 1: 23744 MiB used, 17215 MiB free, 40960 MiB total

--> ... prompt LLM to describe the source code.




----- CODE DESCRIPTION and ANALYSIS -----
 (generated by the LLM)



The provided C code is a CUDA implementation of a numerical simulation, likely for solving fluid dynamics problems using the Navier-Stokes equations. Here's a breakdown of its structure and functionality:

1. **Includes and Definitions**:
   - The code includes necessary headers for standard input/output (`stdio.h`), memory management (`stdlib.h`), mathematics (`math.h`), time functions (`time.h`), and CUDA runtime (`cuda_runtime.h`).

2. **Variable Initialization**:
   - Variables like `nx`, `ny` (grid dimensions) and `Re` (Reynolds number) are initialized.
   - Memory is allocated on both the GPU (using `cudaMalloc`) and CPU for arrays storing velocities, pressure, and intermediate fields.

3. **Time Stepping Loop**:
   - The simulation runs for a maximum of `nstep` iterations.
   - Within each time step, intermediate fields `F` and `G` are computed on the GPU using kernel launches (`computeF`, `computeG`).
   - Synchronization ensures all GPU tasks complete before proceeding.

4. **Pressure Calculation**:
   - Uses Successive Over-Relaxation (SOR) method to solve for pressure.
   - Iterates up to `niter` times, updating the residual and checking convergence against tolerance `Tol`.

5. **Velocity Update**:
   - Velocities are updated using the computed pressure field through kernel launches (`update_u`, `update_v`).
   - Boundary conditions are reapplied after updates.

6. **Boundary Conditions**:
   - Kernel `set_bcs` enforces boundary conditions on velocity fields, crucial for accurate simulation.

7. **Data Output**:
   - After all time steps, velocity data is written to "velocity_gpu.dat".
   - The grid points' velocities are averaged and formatted for output.

8. **Memory Management**:
   - Allocated memory is freed both on GPU and CPU to prevent leaks.

The code efficiently leverages CUDA for parallel computations, minimizing CPU usage to initialization, synchronization, and I/O. Each kernel handles specific tasks, promoting modularity. The use of SOR indicates an iterative approach to solving the pressure field, while boundary condition enforcement ensures physical accuracy. This structure is typical in computational fluid dynamics simulations, optimized for performance on GPU architectures.

... (3 outputs left out)



----- REVIEW THE REFACTORING PLAN PROMPT  (prompt with code, token count: 10793): 

To enhance the energy efficiency of the provided CUDA-accelerated simulation code, consider implementing the following strategies:

### Refactoring Plan for Energy Efficiency

1. **Minimize Synchronization:**
   - **Action:** Remove unnecessary `cudaDeviceSynchronize()` calls unless explicitly required. Use CUDA events or streams to manage kernel dependencies without global synchronization.
   - **Benefit:** Reduces overhead and allows better resource utilization by preventing idle times.

2. **Optimize Memory Management:**
   - **Action:** Preallocate all necessary device memory before entering the main loop to avoid repeated allocations within each iteration.
   - **Benefit:** Reduces allocation overhead, leading to faster execution and lower resource usage.

3. **Reduce Host-Device Data Transfers:**
   - **Action:** Keep data on the GPU throughout computations. Use pinned memory for host allocation

Waiting for the system to settle down...: 100%|██████████████████████████████████████████████████████████████████| 10/10 [00:10<00:00,  1.00s/it]


Execute command: ['./lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_exp02test_betterpower_withcontext', '']

Problem size: 512 x 512 
Time = 0.000477, delt = 4.768372e-04, iter = 1, res = 0.000000e+00
Time = 0.000954, delt = 4.768372e-04, iter = 1, res = 0.000000e+00
Time = 0.001000, delt = 4.632568e-05, iter = 1, res = 0.000000e+00

Total execution time of the iteration loop: 0.850702 (s)

Program executed successfully!

-----CODE GENERATION COMPLETE-----

CODE GENERATION TIME: 5.08 minutes.

Self-correction count: 0



UnboundLocalError: local variable 'exe_output_source' referenced before assignment

### TESTING ONLY

In [ ]:
# energy = lassi.getpower.GetGPUpower.idlepower(10)
# print(energy)

# TESTING ONLY
# Names of the CSV files
'''
files = ['translated_code/ollama_codestral:22b-v0.1-q8_0/promptset_CUDA_to_OMP/generated_OMP_gpu_power_results.csv',
             #'./gpu_power_results_mtdv4.csv',
            ]
# Plot power usage and calculate energy
for file_name in files:
    inference_duration = lassi.getpower.plot_gpu_power(file_name)
    total_energy_joules, ave_power = lassi.getpower.calculate_total_energy(file_name)
    
    # Convert energy units
    total_energy_wh = total_energy_joules / 3600  # Convert from Joules to Watt-hours
    total_energy_kwh = total_energy_wh / 1000  # Convert from Watt-hours to Kilowatt-hours
    
    print(f"Inference duration: {inference_duration:.2f} seconds")
    print(f"Total GPU energy consumption for {file_name}:")
    print(f"  {total_energy_kwh:.6f} kWh")
    print(f"  {total_energy_wh:.4f} Wh")
    print(f"  {total_energy_joules:.2f} J")
    print(f"Average GPU power: {ave_power:.2f} W")
    print("----------------------------------------------------------------------------------------------------------------------------------------")
'''

# TESTING: code execution with power
'''
# exe_filename = "translated_code/ollama_codestral:22b-v0.1-q8_0/promptset_CUDA_to_OMP/atomicCost-cuda_main_translatecode_OMP_exp01test_power_withcontext"
exe_filename = "translated_code/input_codes/HeCBench/lid-driven-cavity/lid-driven-cavity-cuda_main"
# matrix-rotate-cuda_main_translatecode_OMP_exp01test_power_withcontext
input_parameters = [""]
power_measure = True
code_source = "target"
target_lang = "CUDA"

lassi.code_manager.execute_code(exe_filename, input_parameters, power_measure, code_source, target_lang)
'''